# 樣本數據（用戶輸入與對應意圖）


In [1]:
data = {
    "text": [
        "我要訂機票", "預訂飛機票", "幫我查詢航班", "我要訂房", "查詢飯店價格", "推薦一間旅館",
        "現在天氣如何", "查詢今天的氣溫", "這週的天氣預報", "播放音樂", "幫我找一首歌", "推薦音樂"
    ],
    "intent": [
        "book_flight", "book_flight", "check_flight", "book_hotel", "check_hotel", "recommend_hotel",
        "check_weather", "check_weather", "check_weather", "play_music", "search_music", "recommend_music"
    ]
}


# 載入套件

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import BertTokenizer, BertModel
import numpy as np

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to 

# 處理樣本數據

In [3]:
## 轉換標籤為數值
intent_labels = list(set(data["intent"]))
intent_to_idx = {intent: idx for idx, intent in enumerate(intent_labels)}
data["intent_idx"] = [intent_to_idx[intent] for intent in data["intent"]]

# 轉換為 TF-IDF 特徵向量
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data["text"]).toarray()
y = np.array(data["intent_idx"])

# 分割訓練集與測試集
X_train, X_test, y_train, y_test, text_train, text_test = train_test_split(X, y, data["text"], test_size=0.2, random_state=42)

In [4]:
# 建立 Dataset 類別

class IntentDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    

train_dataset = IntentDataset(X_train, y_train)
test_dataset = IntentDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

### 設定 MLP 和 CNN

In [5]:
# MLP
class MLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        return self.model(x)

# CNN
class CNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.fc = nn.Linear(32 * input_size, num_classes)
    
    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

## 調用 BERT

In [6]:
# BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def encode_texts(texts, max_length=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

# 適用於 BERT 的 Dataset
class BERTIntentDataset(Dataset):
    def __init__(self, texts, y):
        self.encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.encodings["input_ids"][idx], self.encodings["attention_mask"][idx], self.y[idx]

    
bert_train_dataset = BERTIntentDataset(text_train, y_train)
bert_test_dataset = BERTIntentDataset(text_test, y_test)

bert_train_loader = DataLoader(bert_train_dataset, batch_size=4, shuffle=True)
bert_test_loader = DataLoader(bert_test_dataset, batch_size=4, shuffle=False)
    


### 設定 BERT 模型

In [7]:
class BERTClassifier(nn.Module):
    def __init__(self, num_classes):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.fc(outputs.pooler_output)

# 訓練和測試函數


In [8]:
def train_and_test(model, train_loader, test_loader, epochs=10, use_bert=False):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            if use_bert:
                input_ids, attention_mask, labels = batch
                outputs = model(input_ids, attention_mask)
            else:
                X_batch, labels = batch
                outputs = model(X_batch)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")
    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            if use_bert:
                input_ids, attention_mask, labels = batch
                outputs = model(input_ids, attention_mask)
            else:
                X_batch, labels = batch
                outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"測試準確率: {correct / total:.2f}")


#### 初始化並訓練 MLP

In [9]:
mlp_model = MLP(input_size=X.shape[1], num_classes=len(intent_labels))
train_and_test(mlp_model, train_loader, test_loader, epochs=30)

Epoch 1, Loss: 2.3515
Epoch 2, Loss: 2.1949
Epoch 3, Loss: 2.1132
Epoch 4, Loss: 2.0051
Epoch 5, Loss: 0.4614
Epoch 6, Loss: 1.3277
Epoch 7, Loss: 0.1520
Epoch 8, Loss: 0.0740
Epoch 9, Loss: 0.7353
Epoch 10, Loss: 0.4281
Epoch 11, Loss: 0.2864
Epoch 12, Loss: 0.0774
Epoch 13, Loss: 0.0161
Epoch 14, Loss: 0.0038
Epoch 15, Loss: 0.0010
Epoch 16, Loss: 0.0042
Epoch 17, Loss: 0.0019
Epoch 18, Loss: 0.0017
Epoch 19, Loss: 0.0015
Epoch 20, Loss: 0.0003
Epoch 21, Loss: 0.0010
Epoch 22, Loss: 0.0009
Epoch 23, Loss: 0.0001
Epoch 24, Loss: 0.0002
Epoch 25, Loss: 0.0007
Epoch 26, Loss: 0.0000
Epoch 27, Loss: 0.0001
Epoch 28, Loss: 0.0006
Epoch 29, Loss: 0.0001
Epoch 30, Loss: 0.0000
測試準確率: 0.00


#### 初始化並訓練 CNN

In [10]:
cnn_model = CNN(input_size=X.shape[1], num_classes=len(intent_labels))
train_and_test(cnn_model, train_loader, test_loader, epochs=30)

Epoch 1, Loss: 2.6539
Epoch 2, Loss: 2.3415
Epoch 3, Loss: 1.8211
Epoch 4, Loss: 0.9061
Epoch 5, Loss: 1.3020
Epoch 6, Loss: 0.5822
Epoch 7, Loss: 0.3654
Epoch 8, Loss: 0.1789
Epoch 9, Loss: 0.4008
Epoch 10, Loss: 0.0466
Epoch 11, Loss: 0.0178
Epoch 12, Loss: 0.0297
Epoch 13, Loss: 0.0053
Epoch 14, Loss: 0.0017
Epoch 15, Loss: 0.0012
Epoch 16, Loss: 0.0015
Epoch 17, Loss: 0.0013
Epoch 18, Loss: 0.0009
Epoch 19, Loss: 0.0007
Epoch 20, Loss: 0.0003
Epoch 21, Loss: 0.0001
Epoch 22, Loss: 0.0001
Epoch 23, Loss: 0.0000
Epoch 24, Loss: 0.0001
Epoch 25, Loss: 0.0000
Epoch 26, Loss: 0.0002
Epoch 27, Loss: 0.0000
Epoch 28, Loss: 0.0000
Epoch 29, Loss: 0.0004
Epoch 30, Loss: 0.0000
測試準確率: 0.00


#### 初始化並訓練 BERT

In [11]:
bert_model = BERTClassifier(num_classes=len(intent_labels))
train_and_test(bert_model, bert_train_loader, bert_test_loader, use_bert=True, epochs=30)

Epoch 1, Loss: 5.5784
Epoch 2, Loss: 17.5871
Epoch 3, Loss: 6.2612
Epoch 4, Loss: 4.6606
Epoch 5, Loss: 3.5883
Epoch 6, Loss: 0.4719
Epoch 7, Loss: 14.4376
Epoch 8, Loss: 2.1711
Epoch 9, Loss: 5.6558
Epoch 10, Loss: 3.6752
Epoch 11, Loss: 3.4667
Epoch 12, Loss: 6.2027
Epoch 13, Loss: 4.7829
Epoch 14, Loss: 0.9274
Epoch 15, Loss: 14.5337
Epoch 16, Loss: 1.2931
Epoch 17, Loss: 0.9958
Epoch 18, Loss: 1.2243
Epoch 19, Loss: 2.9618
Epoch 20, Loss: 0.4972
Epoch 21, Loss: 2.8812
Epoch 22, Loss: 2.5451
Epoch 23, Loss: 1.4548
Epoch 24, Loss: 0.6499
Epoch 25, Loss: 4.7018
Epoch 26, Loss: 5.1074
Epoch 27, Loss: 4.4363
Epoch 28, Loss: 2.5604
Epoch 29, Loss: 1.7218
Epoch 30, Loss: 1.6073
測試準確率: 0.00


## 測試用戶輸入

In [12]:
# user_input = ["幫我找飯店", "播放一首好聽的歌"]
user_input = ["請幫我預訂下週五從台北飛往東京的機票", "請查詢今天從紐約飛往倫敦的航班狀態",\
              "我要預訂下週三至週六在大阪的四星級飯店", "請幫我確認我在巴黎的飯店預訂是否成功",\
             "請推薦曼谷市中心適合家庭入住的飯店", "請查詢明天北京的天氣如何", "請播放周杰倫的《夜曲》",\
             "幫我找一首最近很流行的華語情歌", "請推薦一些適合運動時聽的音樂"]
X_input = vectorizer.transform(user_input).toarray()
X_input_tensor = torch.tensor(X_input, dtype=torch.float32)


### MLP Used

In [13]:
mlp_model.eval()
with torch.no_grad():
    outputs = mlp_model(X_input_tensor)
    _, predicted = torch.max(outputs, 1)

for text, intent_idx in zip(user_input, predicted.tolist()):
    print(f"輸入: {text} -> 預測意圖: {intent_labels[intent_idx]}")

輸入: 請幫我預訂下週五從台北飛往東京的機票 -> 預測意圖: check_weather
輸入: 請查詢今天從紐約飛往倫敦的航班狀態 -> 預測意圖: check_weather
輸入: 我要預訂下週三至週六在大阪的四星級飯店 -> 預測意圖: check_weather
輸入: 請幫我確認我在巴黎的飯店預訂是否成功 -> 預測意圖: check_weather
輸入: 請推薦曼谷市中心適合家庭入住的飯店 -> 預測意圖: check_weather
輸入: 請查詢明天北京的天氣如何 -> 預測意圖: check_weather
輸入: 請播放周杰倫的《夜曲》 -> 預測意圖: check_weather
輸入: 幫我找一首最近很流行的華語情歌 -> 預測意圖: check_weather
輸入: 請推薦一些適合運動時聽的音樂 -> 預測意圖: check_weather


### CNN Used

In [14]:
cnn_model.eval()
with torch.no_grad():
    outputs = cnn_model(X_input_tensor)
    _, predicted = torch.max(outputs, 1)

for text, intent_idx in zip(user_input, predicted.tolist()):
    print(f"輸入: {text} -> 預測意圖: {intent_labels[intent_idx]}")

輸入: 請幫我預訂下週五從台北飛往東京的機票 -> 預測意圖: check_weather
輸入: 請查詢今天從紐約飛往倫敦的航班狀態 -> 預測意圖: check_weather
輸入: 我要預訂下週三至週六在大阪的四星級飯店 -> 預測意圖: check_weather
輸入: 請幫我確認我在巴黎的飯店預訂是否成功 -> 預測意圖: check_weather
輸入: 請推薦曼谷市中心適合家庭入住的飯店 -> 預測意圖: check_weather
輸入: 請查詢明天北京的天氣如何 -> 預測意圖: check_weather
輸入: 請播放周杰倫的《夜曲》 -> 預測意圖: check_weather
輸入: 幫我找一首最近很流行的華語情歌 -> 預測意圖: check_weather
輸入: 請推薦一些適合運動時聽的音樂 -> 預測意圖: check_weather


### BERT Used

In [15]:

encodings = encode_texts(user_input)
X_input_tensor = encodings["input_ids"]  # Tokenized input
attention_mask_tensor = encodings["attention_mask"]  # Attention mask

bert_model.eval()
with torch.no_grad():
    outputs = bert_model(X_input_tensor, attention_mask=attention_mask_tensor)
    _, predicted = torch.max(outputs, 1)

for text, intent_idx in zip(user_input, predicted.tolist()):
    print(f"輸入: {text} -> 預測意圖: {intent_labels[intent_idx]}")

輸入: 請幫我預訂下週五從台北飛往東京的機票 -> 預測意圖: check_weather
輸入: 請查詢今天從紐約飛往倫敦的航班狀態 -> 預測意圖: check_weather
輸入: 我要預訂下週三至週六在大阪的四星級飯店 -> 預測意圖: check_weather
輸入: 請幫我確認我在巴黎的飯店預訂是否成功 -> 預測意圖: check_weather
輸入: 請推薦曼谷市中心適合家庭入住的飯店 -> 預測意圖: check_weather
輸入: 請查詢明天北京的天氣如何 -> 預測意圖: check_weather
輸入: 請播放周杰倫的《夜曲》 -> 預測意圖: check_weather
輸入: 幫我找一首最近很流行的華語情歌 -> 預測意圖: check_weather
輸入: 請推薦一些適合運動時聽的音樂 -> 預測意圖: check_weather
